# 2. Data Preparation

Based on the data understanding phase we now build the cleaned table used by
all the following modules: we drop redundant or uninformative attributes,
impute missing/invalid values and engineer a few derived features.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv("../dataset/DM1_game_dataset.csv")
df.shape

(21925, 46)

## 2.1 Dimensionality reduction

We drop identifiers and free text, attributes that carry (almost) no
information (`NumComments` is always zero, `BestPlayers`/`GoodPlayers`/
`NumImplementations` are predominantly zero or empty), poor-quality attributes
(`LanguageEase` is out of scale in most rows, `Family` is 70% missing) and the
`Rank:*` columns, dominated by the "unranked" sentinel. When manufacturer and
community report the same concept we keep the community version, so `NumWant`
(near-duplicate of `NumWish`) is dropped as well. `NumExpansions` and
`NumAlternates` are only needed later for pattern mining, where they are read
back from the raw table.

In [2]:
rank_cols = [c for c in df.columns if c.startswith("Rank:")]
df = df.drop(columns=["BGGId", "Name", "Description", "LanguageEase",
                      "BestPlayers", "GoodPlayers", "NumComments", "NumWant",
                      "Family", "ImagePath", "NumExpansions", "NumAlternates",
                      "NumImplementations", "IsReimplementation"] + rank_cols)
df.shape

(21925, 24)

## 2.2 Handling missing values

* `ComAgeRec`: NaNs are filled with the manufacturer recommendation when it is
  specified (`MfgAgeRec` uses 0 for "unspecified"); what remains gets the median.
* community playtimes: zeros are invalid and replaced with the median of the
  valid entries;
* `MinPlayers` / `MaxPlayers`: zeros replaced with the mode.

In [3]:
df["ComAgeRec"] = df["ComAgeRec"].fillna(df["MfgAgeRec"].replace(0, np.nan))
df["ComAgeRec"] = df["ComAgeRec"].fillna(df["ComAgeRec"].median())
df.loc[df["ComAgeRec"] == 0, "ComAgeRec"] = df["ComAgeRec"].median()

for col in ["ComMinPlaytime", "ComMaxPlaytime"]:
    df.loc[df[col] == 0, col] = df.loc[df[col] > 0, col].median()

for col in ["MinPlayers", "MaxPlayers"]:
    df.loc[df[col] == 0, col] = df.loc[df[col] > 0, col].mode()[0]

df[["ComAgeRec", "ComMinPlaytime", "ComMaxPlaytime",
    "MinPlayers", "MaxPlayers"]].describe().round(2)

,ComAgeRec,ComMinPlaytime,ComMaxPlaytime,MinPlayers,MaxPlayers
count,21925.00,21925.00,21925.00,21925.00,21925.00
mean,10.02,64.57,92.11,2.01,5.74
std,3.12,443.82,529.45,0.69,15.01
min,2.00,1.00,1.00,1.00,1.00
25%,8.00,20.00,30.00,2.00,4.00
50%,10.00,30.00,45.00,2.00,4.00
75%,12.00,60.00,90.00,2.00,6.00
max,21.00,60000.00,60000.00,10.00,999.00


## 2.3 Derived attributes

* **AvgGameweight** — `GameWeight` and `ComWeight` are perfectly correlated;
  after median-imputing their invalid zeros we merge them into their average.
* **ComAvgPlaytime** — mean of community min and max playtime, plus a `log1p`
  version that removes the strong right skew.
* **PopularityScore** — the popularity counts form one highly correlated block,
  so we compress them into the mean of their min-max-normalized `log1p`
  versions (`NumWant` and `NumComments` were already excluded).
* **CategoryLabel** — compact coding of the `Cat:*` flags, 0 meaning that the
  game is uncategorized/unranked.

In [4]:
for col in ["GameWeight", "ComWeight"]:
    df.loc[df[col] == 0, col] = df.loc[df[col] > 0, col].median()
df["AvgGameweight"] = (df["GameWeight"] + df["ComWeight"]) / 2

df["ComAvgPlaytime"] = (df["ComMinPlaytime"] + df["ComMaxPlaytime"]) / 2
df["ComAvgPlaytime_log"] = np.log1p(df["ComAvgPlaytime"])

pop = ["NumOwned", "NumWish", "NumUserRatings", "NumWeightVotes"]
df["PopularityScore"] = MinMaxScaler().fit_transform(np.log1p(df[pop])).mean(axis=1)

cat_order = ["Cat:Thematic", "Cat:Strategy", "Cat:War", "Cat:Family",
             "Cat:CGS", "Cat:Abstract", "Cat:Party", "Cat:Childrens"]
label = np.zeros(len(df), dtype=int)
for k, col in enumerate(cat_order, start=1):
    label = np.where((label == 0) & (df[col] == 1), k, label)
df["CategoryLabel"] = label

df["CategoryLabel"].value_counts().sort_index()

CategoryLabel
0    11184
1     1224
2     2083
3     3269
4     1823
5      241
6      932
7      434
8      735
Name: count, dtype: int64

The attributes now absorbed into the derived features (manufacturer columns,
playtime bounds, popularity counts and the `Cat:*` flags) can be removed. The
resulting table is saved and reused by clustering, classification, regression
and pattern mining.

In [5]:
df_clean = df[["YearPublished", "MinPlayers", "MaxPlayers", "ComAgeRec",
               "Kickstarted", "AvgGameweight", "ComAvgPlaytime",
               "ComAvgPlaytime_log", "PopularityScore", "CategoryLabel",
               "Rating"]]
df_clean.to_csv("../data/df_clean2.2.csv", index=False)
df_clean.describe().round(3)

,YearPublished,MinPlayers,MaxPlayers,ComAgeRec,Kickstarted,AvgGameweight,ComAvgPlaytime,ComAvgPlaytime_log,PopularityScore,CategoryLabel
count,21925.000,21925.000,21925.000,21925.000,21925.000,21925.000,21925.000,21925.000,21925.000,21925.000
mean,1985.495,2.012,5.739,10.025,0.153,2.128,78.343,3.828,0.348,1.742
std,212.486,0.686,15.007,3.120,0.360,0.793,468.758,0.871,0.142,2.234
min,-3500.000,1.000,1.000,2.000,0.000,1.050,1.000,0.693,0.071,0.000
25%,2001.000,2.000,4.000,8.000,0.000,1.464,25.000,3.258,0.242,0.000
50%,2011.000,2.000,4.000,10.000,0.000,2.074,40.000,3.714,0.316,0.000
75%,2017.000,2.000,6.000,12.000,0.000,2.634,75.000,4.331,0.426,3.000
max,2021.000,10.000,999.000,21.000,1.000,5.101,60000.000,11.002,0.975,8.000


In [6]:
df_clean.head()

,YearPublished,MinPlayers,MaxPlayers,ComAgeRec,Kickstarted,AvgGameweight,ComAvgPlaytime,ComAvgPlaytime_log,PopularityScore,CategoryLabel,Rating
0,2014,2,4,8.0,0,1.94225,30.0,3.433987,0.341847,0,Low
1,2021,2,5,8.0,0,1.11280,17.5,2.917771,0.248770,0,Medium
2,2020,1,5,12.0,0,3.74285,90.0,4.510860,0.371358,0,High
3,2004,2,2,7.0,0,1.36465,20.0,3.044522,0.261154,0,Low
4,2019,2,4,8.0,0,1.56150,17.5,2.917771,0.261691,0,Medium
